<a href="https://colab.research.google.com/github/mdrs-thiago/mdrs-thiago.github.io/blob/main/topicos_a/2027/notebooks/lab04_lstm_atencao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Lab 4 — Classificação de texto: LSTM vs. atenção
**Tópicos Especiais A · Deep Learning · UERJ 2027.1** · Aula 16 · Bloco 4 — Sequências e Atenção

**Objetivo:** comparar TF-IDF + regressão logística, BiLSTM e BiLSTM com atenção no IMDB, e visualizar os pesos de atenção.

| | |
|---|---|
| ✅ **Checkpoint (em sala)** | 60 XP: veja as marcações `✅ CHECKPOINT` |
| 📦 **Entrega (7 dias)** | 180 XP: tabela dos 3 modelos, mapas de atenção em 5 reviews e discussão do custo-benefício. |

> Preencha apenas os trechos marcados com `# TODO`. Antes de começar: **Arquivo → Salvar uma cópia no Drive** e, se indicado, **Ambiente de execução → Alterar tipo → T4 GPU**.

In [ ]:
import random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)

In [ ]:
!pip -q install datasets

## 1. Dados: subamostra fixa do IMDB
Os índices são **salvos em arquivo**: o Lab 5 usa exatamente o mesmo split.

In [ ]:
import json, re, time, torch.nn as nn, matplotlib.pyplot as plt, pandas as pd
from collections import Counter
from datasets import load_dataset

imdb = load_dataset('imdb')
rng = np.random.default_rng(42)
tr_idx = rng.choice(len(imdb['train']), 10_000, replace=False).tolist()
va_idx = rng.choice(len(imdb['test']), 2_500, replace=False).tolist()
json.dump({'train': tr_idx, 'val': va_idx}, open('split_lab4.json', 'w'))   # ⚠️ guarde este arquivo para o Lab 5!

X_tr = [imdb['train'][i]['text'] for i in tr_idx]; y_tr = [imdb['train'][i]['label'] for i in tr_idx]
X_va = [imdb['test'][i]['text'] for i in va_idx];  y_va = [imdb['test'][i]['label'] for i in va_idx]
print(len(X_tr), len(X_va), np.mean(y_tr))

## 2. Baseline: TF-IDF + regressão logística

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

t0 = time.time()
# TODO: pipeline TF-IDF (ngram_range=(1, 2), min_df=2) + LogisticRegression(max_iter=1000)
clf = ...
clf.fit(X_tr, y_tr)
p = clf.predict(X_va)
res = [{'modelo': 'TF-IDF + LogReg', 'acc': accuracy_score(y_va, p), 'f1': f1_score(y_va, p), 'tempo (s)': round(time.time() - t0, 1)}]
res

## 3. Tokenização, vocabulário e DataLoader com padding

In [ ]:
MAX_LEN, MAX_VOCAB = 256, 20_000
def tokenize(s):
    s = re.sub(r'<br\s*/?>', ' ', s.lower())
    return re.findall(r"[a-z0-9']+|[!?.,]", s)

cnt = Counter(t for s in X_tr for t in tokenize(s))
itos = ['<pad>', '<unk>'] + [w for w, _ in cnt.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}

def encode(s):
    # TODO: tokenize, converta para ids (use 1 para palavras fora do vocabulário) e trunque em MAX_LEN
    return ...

class TextDS(torch.utils.data.Dataset):
    def __init__(self, X, y): self.ids = [torch.tensor(encode(s)) for s in X]; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.ids[i], self.y[i]

def collate(batch):
    ids, y = zip(*batch)
    lengths = torch.tensor([len(t) for t in ids])
    ids = nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=0)
    return ids, lengths, torch.tensor(y)

tr_loader = torch.utils.data.DataLoader(TextDS(X_tr, y_tr), batch_size=64, shuffle=True, collate_fn=collate)
va_loader = torch.utils.data.DataLoader(TextDS(X_va, y_va), batch_size=128, collate_fn=collate)
ids, lengths, yb = next(iter(tr_loader)); print(ids.shape, lengths[:8], yb[:8])

### ✅ CHECKPOINT
Mostre ao professor: o arquivo `split_lab4.json`, o resultado do baseline e o shape de um batch.

## 4. BiLSTM (many-to-one)

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTM(nn.Module):
    def __init__(self, V, d=128, h=128, n_cls=2):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.lstm = nn.LSTM(d, h, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(2 * h, n_cls)
    def forward(self, ids, lengths):
        x = self.emb(ids)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        # TODO: concatene o último estado da direção forward (h_n[-2]) e da backward (h_n[-1])
        h = ...
        return self.fc(h)

## 5. BiLSTM + atenção (com máscara de padding)

In [ ]:
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.proj = nn.Linear(d, d); self.q = nn.Parameter(torch.randn(d) / d ** 0.5)
    def forward(self, H, mask):                       # H: (B, T, d), mask: (B, T) True = token real
        scores = torch.tanh(self.proj(H)) @ self.q    # (B, T)
        # TODO: mascare os pads com -1e9, aplique softmax em T e calcule o contexto ponderado
        alpha = ...
        ctx = ...
        return ctx, alpha

class BiLSTMAttn(nn.Module):
    def __init__(self, V, d=128, h=128, n_cls=2):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.lstm = nn.LSTM(d, h, batch_first=True, bidirectional=True)
        self.attn = AttnPool(2 * h); self.fc = nn.Linear(2 * h, n_cls)
    def forward(self, ids, lengths, return_attn=False):
        x = self.emb(ids)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        H, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=ids.shape[1])
        ctx, alpha = self.attn(H, ids != 0)
        out = self.fc(ctx)
        return (out, alpha) if return_attn else out

## 6. Treino e comparação

In [ ]:
def train_text(model, epochs=4, lr=2e-3):
    model.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr); lf = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for ids, lengths, yb in tr_loader:
            ids, yb = ids.to(device), yb.to(device)
            loss = lf(model(ids, lengths), yb)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        acc, f1 = eval_text(model); print(f'ep {ep+1} | val acc {acc:.3f} f1 {f1:.3f}')
    return time.time() - t0

@torch.no_grad()
def eval_text(model):
    model.eval(); ps = []
    for ids, lengths, yb in va_loader: ps += model(ids.to(device), lengths).argmax(1).cpu().tolist()
    return accuracy_score(y_va, ps), f1_score(y_va, ps)

for name, cls in [('BiLSTM', BiLSTM), ('BiLSTM + atenção', BiLSTMAttn)]:
    set_seed(42); m = cls(len(itos)); t = train_text(m); acc, f1 = eval_text(m)
    res.append({'modelo': name, 'acc': acc, 'f1': f1, 'tempo (s)': round(t, 1), 'params': sum(p.numel() for p in m.parameters())})
    if name.endswith('atenção'): attn_model = m
pd.DataFrame(res)

## 7. Visualizando a atenção

In [ ]:
from IPython.display import HTML, display
def show_attention(text, label):
    ids = torch.tensor([encode(text)]); L = torch.tensor([ids.shape[1]])
    attn_model.eval()
    with torch.no_grad(): out, alpha = attn_model(ids.to(device), L, return_attn=True)
    a = alpha[0].cpu().numpy(); a = a / a.max(); toks = [itos[i] for i in ids[0]]
    spans = ''.join(f'<span style="background: rgba(0,75,135,{w:.2f}); color: {"white" if w > .5 else "black"}; padding:1px">{t}</span> ' for t, w in zip(toks, a))
    pred = out.argmax(1).item()
    display(HTML(f'<p><b>real={label} pred={pred}</b><br>{spans}</p>'))

# TODO: escolha 5 reviews da validação (≥ 2 acertos e ≥ 2 erros) e visualize
for k in range(5):
    show_attention(X_va[k][:1200], y_va[k])

## 8. Análise (📦 entrega)
- O baseline perdeu por quanto? Valeu o custo extra (tempo, parâmetros, complexidade)?
- A atenção se concentra onde você esperava (adjetivos, negações, "but")?
- Nos erros, o que a atenção revela?